In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
import warnings

warnings.filterwarnings("ignore", category=UserWarning)




In [2]:
class Config:
    tabular_columns = [
        "Subject Focus",
        "Eyes",
        "Face",
        "Near",
        "Action",
        "Accessory",
        "Group",
        "Collage",
        "Human",
        "Occlusion",
        "Info",
        "Blur",
    ]
    n_splits = 5
    random_state = 997
    n_estimators = 300
    learning_rate = 0.05
    max_depth = 3
    subsample = 0.8
    # Updated to a valid loss name for scikit‑learn ≥1.2
    loss = "squared_error"  # least squares regression


config = Config()



In [3]:
train_path = "../input/petfinder-pawpularity-score/train.csv"
test_path = "../input/petfinder-pawpularity-score/test.csv"
sample_sub_path = "../input/petfinder-pawpularity-score/sample_submission.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

train_df["file_path"] = (
    "../input/petfinder-pawpularity-score/train/" + train_df["Id"] + ".jpg"
)
test_df["file_path"] = (
    "../input/petfinder-pawpularity-score/test/" + test_df["Id"] + ".jpg"
)




In [4]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))




In [5]:
kf = KFold(n_splits=config.n_splits, shuffle=True, random_state=config.random_state)

fold_val_scores = []
models = []

for fold_idx, (train_idx, val_idx) in enumerate(kf.split(train_df)):
    X_train = train_df.loc[train_idx, Config.tabular_columns].values.astype(float)
    y_train = train_df.loc[train_idx, "Pawpularity"].values.astype(float)
    X_val = train_df.loc[val_idx, Config.tabular_columns].values.astype(float)
    y_val = train_df.loc[val_idx, "Pawpularity"].values.astype(float)

    gbr = GradientBoostingRegressor(
        n_estimators=config.n_estimators,
        learning_rate=config.learning_rate,
        max_depth=config.max_depth,
        subsample=config.subsample,
        loss=config.loss,
        random_state=config.random_state,
    )
    gbr.fit(X_train, y_train)

    val_pred = gbr.predict(X_val)
    score = rmse(y_val, val_pred)
    fold_val_scores.append(score)

    print(f"Fold {fold_idx + 1} validation RMSE: {score:.4f}")

    models.append(gbr)

print(f"\nAverage validation RMSE: {np.mean(fold_val_scores):.4f}")



Fold 1 validation RMSE: 20.5495
Fold 2 validation RMSE: 19.8424
Fold 3 validation RMSE: 21.5357
Fold 4 validation RMSE: 20.5599
Fold 5 validation RMSE: 20.9812

Average validation RMSE: 20.6937


In [6]:
X_test = test_df[Config.tabular_columns].values.astype(float)

# Average predictions from all folds
test_preds = np.mean([m.predict(X_test) for m in models], axis=0)

# Ensure predictions respect competition bounds and format
test_preds = np.clip(test_preds, 1, 100)
test_preds = np.round(test_preds, 2)  # two‑decimal precision as in sample submission



In [7]:
sample_submission = pd.read_csv(sample_sub_path)
sample_submission["Pawpularity"] = test_preds
submission_path = "submission.csv"
sample_submission.to_csv(submission_path, index=False)
print(f"Submission file written to {submission_path}")

Submission file written to submission.csv
